# Notebook 04 - LangGraph Agent

**Project:** IncidentIQ - AI-powered Incident Intelligence

**Goal:** Wire all five tools from Notebook 03 into a LangGraph agent with a router node,
conversation memory and LangSmith tracing.

## What this notebook covers
1. Install and import all required packages
2. Load all tools from Notebook 03
3. Configure LangSmith tracing
4. Build the LangGraph agent graph
5. Add conversation memory
6. Test the agent with natural language requests
7. Run quality tests

## What makes this a real agent
A real agent is not a fixed pipeline. It reasons about what the user wants
and decides autonomously which tools to call, in which order, and how many times.

Examples of autonomous decisions:
- User drops a URL and asks a question -> agent calls fetch tool then search tool
- User asks for a cheatsheet and emails -> agent calls PDF tool then Gmail tool
- User asks a follow-up question -> agent uses memory to maintain context

---

## 1. Install required packages

Installing LangGraph and LangSmith on top of the packages from Notebook 03.
Run this cell once - skip on subsequent runs.

In [ ]:
!pip install langgraph langsmith langchain langchain-openai langchain-community chromadb youtube-transcript-api reportlab google-api-python-client google-auth-oauthlib python-dotenv -q
print('Packages installed.')

## 2. Import libraries and load environment variables

We import LangGraph components on top of everything from Notebook 03.
LangSmith tracing is enabled via environment variables - no code changes needed.

In [ ]:
import os
import re
import base64
import json
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv
from typing import Annotated

# LangChain
from langchain.tools import tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema.output_parser import StrOutputParser
from langchain.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage

# LangGraph
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

# YouTube
from youtube_transcript_api import YouTubeTranscriptApi, NoTranscriptFound, TranscriptsDisabled

# PDF
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm
from reportlab.lib.colors import HexColor, white
from reportlab.pdfgen import canvas

# Gmail
from googleapiclient.discovery import build
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders

import chromadb

load_dotenv()
assert os.getenv('OPENAI_API_KEY'), 'OPENAI_API_KEY not found. Check your .env file.'
print('Environment variables loaded successfully.')

## 3. Configuration

Central configuration shared across all components.
Must match the values used in Notebook 01, 02 and 03.

In [ ]:
# Must match Notebook 01, 02 and 03
CHROMA_PATH      = '../chroma_db'
COLLECTION_NAME  = 'incidentiq'
EMBEDDING_MODEL  = 'text-embedding-3-small'
LLM_MODEL        = 'gpt-4o-mini'
RETRIEVER_K      = 8

# PDF colors
RED    = HexColor('#C0392B')
DARK   = HexColor('#1C2833')
ORANGE = HexColor('#E67E22')
GREEN  = HexColor('#1E8449')
WHITE  = white

# Gmail
DISTRIBUTION_LIST = os.getenv('GMAIL_DISTRIBUTION_LIST', '').split(',')
GMAIL_SCOPES      = ['https://www.googleapis.com/auth/gmail.send']

# Initialize shared components
llm             = ChatOpenAI(model=LLM_MODEL, temperature=0)
embedding_model = OpenAIEmbeddings(model=EMBEDDING_MODEL)
chroma_client   = chromadb.PersistentClient(path=CHROMA_PATH)
vectorstore     = Chroma(
    client=chroma_client,
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_model,
)

print('Configuration set.')
print(f'   LLM model:       {LLM_MODEL}')
print(f'   ChromaDB path:   {CHROMA_PATH}')

## 4. Configure LangSmith tracing

LangSmith automatically logs every agent step - tool calls, LLM calls, decisions and timing.
This requires only environment variables - no changes to the agent code.

What LangSmith shows during the demo:
- Which node was active at each step
- Which tool the agent chose and why
- How long each step took
- Token usage and cost per run
- The full conversation history

Open smith.langchain.com during the demo to show the jury the agent thinking in real time.

In [ ]:
# Enable LangSmith tracing - logs all agent steps automatically
# Requires LANGSMITH_API_KEY in your .env file
# Get a free key at smith.langchain.com
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_PROJECT']    = 'incidentiq-agent'

if os.getenv('LANGSMITH_API_KEY'):
    os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
    print('LangSmith tracing enabled.')
    print('   View traces at: https://smith.langchain.com')
    print('   Project: incidentiq-agent')
else:
    print('LangSmith API key not found - tracing disabled.')
    print('Add LANGSMITH_API_KEY to your .env file for tracing.')
    print('Get a free key at smith.langchain.com')

## 5. Load all tools

We redefine all five tools here so this notebook is self-contained.
The tool definitions are identical to Notebook 03.
The @tool decorator makes each function callable by the LangGraph agent.

In [ ]:
# Helper functions shared across tools
def extract_video_id(url):
    if 'v=' in url:
        return url.split('v=')[1].split('&')[0]
    elif 'youtu.be/' in url:
        return url.split('youtu.be/')[1].split('?')[0]
    raise ValueError(f'Cannot extract video ID from URL: {url}')

def clean_transcript(text):
    text = re.sub(r'\[Music\]|\[Applause\]|\[Laughter\]|\[Cheering\]', '', text)
    text = re.sub(r'\[\d{2}:\d{2}\]\s*(?=\[\d{2}:\d{2}\])', '', text)
    return re.sub(r'\s+', ' ', text).strip()

def extract_keypoints_for_pdf(context, language='dutch'):
    lang_map = {'dutch': 'Dutch', 'english': 'English', 'french': 'French'}
    lang = lang_map.get(language.lower(), 'Dutch')
    prompt = (
        f'Extract structured information from the context below for an incident training cheatsheet.\n'
        f'Respond in {lang} with this exact JSON format and nothing else:\n'
        f'{{"title": "Short title", "subtitle": "Source name", '
        f'"keypoints": ["point 1", "point 2", "point 3", "point 4", "point 5"], '
        f'"recommendations": ["rec 1", "rec 2", "rec 3", "rec 4"]}}\n\n'
        f'Rules: max 12 words per item, simple language, no timestamps.\n\n'
        f'Context:\n{context}\n\nJSON:'
    )
    response = llm.invoke(prompt)
    raw = re.sub(r'```json|```', '', response.content.strip()).strip()
    return json.loads(raw)

def generate_pdf(data, source_url=''):
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filepath  = f'/tmp/incidentiq_cheatsheet_{timestamp}.pdf'
    c = canvas.Canvas(filepath, pagesize=A4)
    W, H = A4
    c.setFillColor(RED)
    c.rect(0, H-3.2*cm, W, 3.2*cm, fill=1, stroke=0)
    c.setFillColor(WHITE)
    c.circle(1.8*cm, H-1.6*cm, 0.85*cm, fill=1, stroke=0)
    c.setFillColor(RED)
    c.setFont('Helvetica-Bold', 14)
    c.drawCentredString(1.8*cm, H-1.95*cm, 'IQ')
    c.setFillColor(WHITE)
    c.setFont('Helvetica-Bold', 15)
    c.drawString(3.2*cm, H-1.3*cm, data.get('title', 'IncidentIQ Cheatsheet'))
    c.setFont('Helvetica', 10)
    c.drawString(3.2*cm, H-1.85*cm, data.get('subtitle', ''))
    c.setFont('Helvetica', 8)
    c.drawRightString(W-1.2*cm, H-1.3*cm, datetime.now().strftime('%d/%m/%Y'))
    c.drawRightString(W-1.2*cm, H-1.75*cm, 'Generated by IncidentIQ AI')
    c.setFillColor(ORANGE)
    c.rect(0, H-3.6*cm, W, 0.4*cm, fill=1, stroke=0)
    y = H-5.0*cm
    def sh(y, title, color=DARK):
        c.setFillColor(color)
        c.setFont('Helvetica-Bold', 11)
        c.drawString(1.2*cm, y, title.upper())
        c.setStrokeColor(color)
        c.setLineWidth(1.5)
        c.line(1.2*cm, y-0.2*cm, W-1.2*cm, y-0.2*cm)
        return y-0.7*cm
    def bi(y, text, color=DARK, bc=RED):
        c.setFillColor(bc)
        c.circle(1.2*cm, y+0.2*cm, 0.1*cm, fill=1, stroke=0)
        c.setFillColor(color)
        c.setFont('Helvetica', 9.5)
        mw = W-1.5*cm-1.2*cm
        words = text.split()
        line, lines = '', []
        for w in words:
            t = line+w+' '
            if c.stringWidth(t, 'Helvetica', 9.5) < mw:
                line = t
            else:
                lines.append(line.strip())
                line = w+' '
        lines.append(line.strip())
        for i, l in enumerate(lines):
            c.drawString(1.5*cm, y-i*0.45*cm, l)
        return y-len(lines)*0.45*cm-0.3*cm
    y = sh(y, 'Key Points', RED)
    for kp in data.get('keypoints', []):
        y = bi(y, kp)
    y -= 0.4*cm
    y = sh(y, 'AI-generated recommendations based on the video', GREEN)
    c.setFillColor(HexColor('#EAFAF1'))
    c.setStrokeColor(GREEN)
    c.setLineWidth(0.8)
    c.roundRect(1.2*cm, y-0.75*cm, W-2.4*cm, 0.65*cm, 3, fill=1, stroke=1)
    c.setFillColor(GREEN)
    c.setFont('Helvetica-Bold', 8)
    c.drawString(1.55*cm, y-0.3*cm, 'AI analysis:')
    c.setFillColor(DARK)
    c.setFont('Helvetica-Oblique', 8)
    c.drawString(3.0*cm, y-0.3*cm, 'Automatically extracted from video - not cited from a person.')
    y -= 1.0*cm
    for rec in data.get('recommendations', []):
        y = bi(y, rec, bc=GREEN)
    c.setFillColor(RED)
    c.rect(0, 1.2*cm, W, 0.15*cm, fill=1, stroke=0)
    c.setFillColor(DARK)
    c.rect(0, 0, W, 1.2*cm, fill=1, stroke=0)
    c.setFillColor(WHITE)
    c.setFont('Helvetica', 7.5)
    c.drawString(1.2*cm, 0.65*cm, 'IncidentIQ - AI-powered Incident Intelligence')
    if source_url:
        c.drawCentredString(W/2, 0.65*cm, f'Source: {source_url[:70]}')
    c.drawRightString(W-1.2*cm, 0.65*cm, 'Page 1/1')
    c.save()
    return filepath

def get_gmail_service():
    creds = None
    token_path = Path('../token.json')
    creds_path = Path('../credentials.json')
    assert creds_path.exists(), 'credentials.json not found in project root.'
    if token_path.exists():
        creds = Credentials.from_authorized_user_file(str(token_path), GMAIL_SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(str(creds_path), GMAIL_SCOPES)
            creds = flow.run_local_server(port=0)
        token_path.write_text(creds.to_json())
    return build('gmail', 'v1', credentials=creds)

def create_email_with_attachment(sender, recipients, subject, body, pdf_path):
    msg = MIMEMultipart()
    msg['From']    = sender
    msg['To']      = ', '.join(recipients)
    msg['Subject'] = subject
    msg.attach(MIMEText(body, 'plain'))
    with open(pdf_path, 'rb') as f:
        part = MIMEBase('application', 'octet-stream')
        part.set_payload(f.read())
    encoders.encode_base64(part)
    part.add_header('Content-Disposition', f'attachment; filename={Path(pdf_path).name}')
    msg.attach(part)
    return {'raw': base64.urlsafe_b64encode(msg.as_bytes()).decode()}

print('Helper functions loaded.')

In [ ]:
# Define all five agent tools
# These are identical to Notebook 03 - redefined here for self-contained notebook

@tool
def fetch_youtube_transcript(youtube_url: str) -> str:
    """
    Fetch the transcript of a YouTube video and store it in ChromaDB.
    Use this tool when the user provides a YouTube URL.
    Supports English, Dutch and French transcripts.
    Works with any incident training video that has subtitles enabled.
    Returns a confirmation with the number of chunks stored.
    """
    try:
        video_id = extract_video_id(youtube_url)
        try:
            entries = YouTubeTranscriptApi().fetch(video_id, languages=['en', 'nl', 'fr'])
            transcript_list = entries.snippets
        except NoTranscriptFound:
            return f'No transcript found for video {video_id}. Please use a video with CC subtitles.'
        except TranscriptsDisabled:
            return f'Transcripts are disabled for video {video_id}.'
        plain = ' '.join(t.text for t in transcript_list)
        timestamped = ' '.join(
            f'[{int(t.start//60):02d}:{int(t.start%60):02d}] {t.text}'
            for t in transcript_list
        )
        plain       = clean_transcript(plain)
        timestamped = clean_transcript(timestamped)
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=500, chunk_overlap=50, separators=['\n\n', '\n', '. ', ' ']
        )
        chunks = splitter.create_documents(
            texts=[timestamped],
            metadatas=[{'video_id': video_id, 'source': youtube_url}]
        )
        existing = vectorstore.get(where={'video_id': video_id})
        if existing['ids']:
            vectorstore.delete(ids=existing['ids'])
        vectorstore.add_documents(chunks)
        return (
            f'Transcript loaded successfully.\n'
            f'Video ID: {video_id}\n'
            f'Total characters: {len(plain):,}\n'
            f'Chunks stored in ChromaDB: {len(chunks)}\n'
            f'Ready for Q&A.'
        )
    except Exception as e:
        return f'Error fetching transcript: {str(e)}'


@tool
def search_video_knowledge(query: str) -> str:
    """
    Search the ChromaDB knowledge base for information relevant to the query.
    Use this tool to answer questions about the loaded video content.
    Automatically translates non-English queries for better retrieval quality.
    Works with questions in any language about any incident training video.
    Returns the most relevant transcript excerpts with timestamp sources.
    """
    try:
        english_query = llm.invoke(
            f'Translate to English, return only the translation: {query}'
        ).content.strip()
        results = vectorstore.similarity_search(english_query, k=RETRIEVER_K)
        if not results:
            return 'No relevant information found. Please load a YouTube video first.'
        all_ts = re.findall(r'\[\d{2}:\d{2}\]', ' '.join([r.page_content for r in results]))
        seen, unique_ts = set(), []
        for t in all_ts:
            if t not in seen:
                seen.add(t)
                unique_ts.append(t)
        clean_chunks = [
            re.sub(r'\[\d{2}:\d{2}\]\s*(?=\[\d{2}:\d{2}\])', '', r.page_content)
            for r in results
        ]
        return '\n\n'.join(clean_chunks) + f'\n\nSources: {" | ".join(unique_ts[:5])}'
    except Exception as e:
        return f'Error searching knowledge base: {str(e)}'


@tool
def summarize_video(language: str = 'english') -> str:
    """
    Generate a structured summary of the entire loaded video.
    Use this tool when the user asks for a full summary or overview of the video.
    Specify language as 'english', 'dutch' or 'french'.
    Works with any incident training video loaded into ChromaDB.
    Returns a structured summary with introduction, key points, lessons and conclusion.
    """
    try:
        results = vectorstore.similarity_search(
            'main topic lessons learned conclusions key points', k=12
        )
        if not results:
            return 'No video content found. Please load a YouTube video first.'
        context = '\n\n'.join([
            re.sub(r'\[\d{2}:\d{2}\]\s*(?=\[\d{2}:\d{2}\])', '', r.page_content)
            for r in results
        ])
        lang_map = {
            'english': 'English',
            'dutch':   'Dutch - use natural direct language as used in Belgian incident training',
            'french':  'French',
        }
        lang = lang_map.get(language.lower(), 'English')
        prompt = (
            f'You are an incident training expert. Write a structured summary in {lang}.\n'
            f'Structure: **Introduction** / **Key Points** / **Lessons Learned** / **Conclusion**\n'
            f'Rules: bullet points only, max 15 words per bullet, based strictly on context.\n\n'
            f'Context:\n{context}\n\nSummary:'
        )
        return llm.invoke(prompt).content.strip()
    except Exception as e:
        return f'Error generating summary: {str(e)}'


@tool
def generate_pdf_cheatsheet(language: str = 'dutch', source_url: str = '') -> str:
    """
    Generate a professional 1-page PDF cheatsheet from the loaded video content.
    Use this tool when the user asks for a cheatsheet, key concepts document or PDF summary.
    Specify language as 'dutch', 'english' or 'french'.
    Works with any incident training video loaded into ChromaDB.
    Returns the file path of the generated PDF for the Gmail tool to attach.
    """
    try:
        results = vectorstore.similarity_search(
            'key points lessons learned recommendations conclusions', k=10
        )
        if not results:
            return 'No video content found. Please load a YouTube video first.'
        context = '\n\n'.join([r.page_content for r in results])
        data     = extract_keypoints_for_pdf(context, language)
        filepath = generate_pdf(data, source_url)
        return (
            f'PDF cheatsheet generated successfully.\n'
            f'File path: {filepath}\n'
            f'Title: {data.get("title", "N/A")}'
        )
    except Exception as e:
        return f'Error generating PDF: {str(e)}'


@tool
def send_gmail_with_cheatsheet(pdf_path: str, custom_emails: str = '') -> str:
    """
    Send the generated PDF cheatsheet to the distribution list via Gmail.
    Use this tool after generate_pdf_cheatsheet has created the PDF file.
    pdf_path: the file path returned by the generate_pdf_cheatsheet tool.
    custom_emails: optional comma-separated email addresses added to the send list.
    This allows users to type their own email address in the Streamlit interface.
    Works for any organization - fire service, police, EMS or civil protection.
    Returns a confirmation message with the full list of recipients.
    """
    try:
        assert Path(pdf_path).exists(), f'PDF not found at path: {pdf_path}'
        recipients = [e.strip() for e in DISTRIBUTION_LIST if e.strip()]
        if custom_emails:
            recipients.extend([e.strip() for e in custom_emails.split(',') if e.strip()])
        assert recipients, 'No recipients found. Add emails to GMAIL_DISTRIBUTION_LIST in .env.'
        service = get_gmail_service()
        subject = f'IncidentIQ - Key Concepts Cheatsheet - {datetime.now().strftime("%d/%m/%Y")}'
        body = (
            'Dear colleague,\n\n'
            'Please find attached the AI-generated key concepts cheatsheet '
            'from the latest incident training video.\n\n'
            'Generated by IncidentIQ AI Agent.\n'
            'Recommendations should be reviewed by a qualified officer before operational use.\n\n'
            'Best regards,\nIncidentIQ AI Agent'
        )
        message = create_email_with_attachment('me', recipients, subject, body, pdf_path)
        service.users().messages().send(userId='me', body=message).execute()
        return (
            f'Cheatsheet sent successfully.\n'
            f'Recipients: {", ".join(recipients)}\n'
            f'Subject: {subject}'
        )
    except Exception as e:
        return f'Error sending email: {str(e)}'


# Collect all tools
AGENT_TOOLS = [
    fetch_youtube_transcript,
    search_video_knowledge,
    summarize_video,
    generate_pdf_cheatsheet,
    send_gmail_with_cheatsheet,
]

print(f'All {len(AGENT_TOOLS)} tools loaded and ready.')
for i, t in enumerate(AGENT_TOOLS, 1):
    print(f'   Tool {i}: {t.name}')

## 6. Define the agent system prompt

The system prompt tells the agent who it is, what tools it has and how to behave.
This is the most important configuration for agent quality.

Key rules:
- Always respond in the user language
- Use tools autonomously based on user intent
- Never make up information not in the video
- Always confirm what was done after tool calls

In [ ]:
AGENT_SYSTEM_PROMPT = """You are IncidentIQ, an AI agent specialized in incident training and knowledge extraction.
You help emergency services professionals extract knowledge from incident training videos.

You work for any organization: fire services, police, EMS, civil protection.

You have access to these tools:
- fetch_youtube_transcript: load a YouTube video into the knowledge base
- search_video_knowledge: answer questions about the loaded video
- summarize_video: generate a structured summary of the video
- generate_pdf_cheatsheet: create a 1-page PDF with key concepts
- send_gmail_with_cheatsheet: send the PDF to email addresses

HOW TO BEHAVE:
- When you receive a YouTube URL: immediately call fetch_youtube_transcript
- When the user asks a question about the video: call search_video_knowledge
- When the user asks for a summary: call summarize_video
- When the user asks for a cheatsheet or key concepts: call generate_pdf_cheatsheet
- When the user asks to send an email: call send_gmail_with_cheatsheet
- When multiple actions are needed: chain the tools in the correct order

LANGUAGE RULE:
- Always respond in the same language as the user message
- Dutch question -> Dutch answer
- English question -> English answer
- French question -> French answer

ANSWER FORMAT:
- Use bullet points - never long paragraphs
- Keep bullets short - maximum 15 words each
- After tool calls: confirm clearly what was done
- Never make up information not present in the video
"""

print('System prompt defined.')
print(f'   Length: {len(AGENT_SYSTEM_PROMPT)} characters')

## 7. Build the LangGraph agent graph

The LangGraph graph defines how the agent thinks and acts.
It has two nodes connected by conditional edges:

- agent node: the LLM that reads messages and decides what to do next
- tools node: executes whichever tool the agent selected

The router (tools_condition) checks after each agent step:
- If the agent wants to call a tool -> go to tools node
- If the agent is done -> end the conversation

Memory (MemorySaver) stores the full conversation between turns
so the agent remembers context across multiple messages.

In [ ]:
# Bind all tools to the LLM
# This tells the LLM which tools are available and what their signatures are
llm_with_tools = llm.bind_tools(AGENT_TOOLS)


def agent_node(state: MessagesState):
    """
    Agent node - the brain of the agent.
    Reads all messages in the current state, including conversation history,
    and decides whether to call a tool or respond directly to the user.
    The system prompt is prepended to every call to maintain consistent behavior.
    """
    system = SystemMessage(content=AGENT_SYSTEM_PROMPT)
    messages = [system] + state['messages']
    response = llm_with_tools.invoke(messages)
    return {'messages': [response]}


# Build the graph
builder = StateGraph(MessagesState)

# Add nodes
builder.add_node('agent', agent_node)           # LLM reasoning node
builder.add_node('tools', ToolNode(AGENT_TOOLS)) # Tool execution node

# Add edges
# START -> agent: always begin with the agent reasoning
builder.add_edge(START, 'agent')

# agent -> tools or END: conditional routing based on agent decision
# tools_condition checks if the agent wants to call a tool
builder.add_conditional_edges('agent', tools_condition)

# tools -> agent: after tool execution, return to agent for next decision
builder.add_edge('tools', 'agent')

# Compile with memory
# MemorySaver stores conversation history between turns using thread_id
memory = MemorySaver()
agent  = builder.compile(checkpointer=memory)

print('LangGraph agent compiled successfully.')
print('   Nodes: agent, tools')
print('   Memory: MemorySaver (in-memory, per thread_id)')
print('   LangSmith: tracing all steps automatically')

## 8. Visualize the agent graph

LangGraph can generate a visual diagram of the agent graph.
This shows exactly how nodes and edges are connected.
Useful for documentation and the jury presentation.

In [ ]:
# Visualize the graph structure
# Requires graphviz or mermaid - falls back to text representation if not available
try:
    from IPython.display import Image, display
    display(Image(agent.get_graph().draw_mermaid_png()))
except Exception:
    # Text fallback if visualization libraries are not available
    print('Graph structure:')
    print('  START -> agent_node')
    print('  agent_node -> tools_node  (if tool call detected)')
    print('  agent_node -> END         (if response is final)')
    print('  tools_node -> agent_node  (always - loop back for next decision)')

## 9. Build the ask() function

The ask() function is the single entry point for all interactions with the agent.
It handles the thread_id for memory, streams the response and formats the output.

thread_id: a unique identifier per conversation session.
Using the same thread_id across calls gives the agent memory of previous messages.
Using a new thread_id starts a fresh conversation.

In [ ]:
def ask(
    message: str,
    thread_id: str = 'default',
    verbose: bool = False,
) -> str:
    """
    Send a message to the IncidentIQ agent and get a response.
    The agent autonomously decides which tools to call based on the message.

    Args:
        message:   Natural language message in any language
        thread_id: Conversation session ID - same ID = memory across turns
        verbose:   If True, show all intermediate steps and tool calls

    Returns:
        The agent final response as a string
    """
    config = {'configurable': {'thread_id': thread_id}}
    inputs = {'messages': [HumanMessage(content=message)]}

    if verbose:
        print(f'\nProcessing: "{message}"')
        print('-' * 50)

    # Stream all events - collect tool calls and final response
    final_response = ''
    tool_calls_made = []

    for event in agent.stream(inputs, config=config, stream_mode='values'):
        last_message = event['messages'][-1]

        # Track tool calls for verbose output
        if verbose and hasattr(last_message, 'tool_calls') and last_message.tool_calls:
            for tc in last_message.tool_calls:
                tool_calls_made.append(tc['name'])
                print(f'   Calling tool: {tc["name"]}')

        # Capture final AI response
        if hasattr(last_message, 'content') and isinstance(last_message.content, str):
            if last_message.content.strip():
                final_response = last_message.content.strip()

    if verbose and tool_calls_made:
        print(f'   Tools used: {" -> ".join(tool_calls_made)}')
        print('-' * 50)

    return final_response


print('ask() function ready.')
print('   Usage: ask("your message here")')
print('   Memory: ask("follow-up", thread_id="session1")')

## 10. Test the agent - single tool calls

We test the agent with simple requests that each require one tool call.
This verifies that the router correctly identifies user intent and calls the right tool.

In [ ]:
print('=' * 60)
print('TEST 1 - Load YouTube video')
print('=' * 60)
result = ask(
    'https://www.youtube.com/watch?v=7OH5FEWWM_k',
    thread_id='test1',
    verbose=True
)
print(f'Agent: {result}')

In [ ]:
print('=' * 60)
print('TEST 2 - Question about the video in English')
print('=' * 60)
result = ask(
    'What mistakes were made during the incident?',
    thread_id='test2',
    verbose=True
)
print(f'Agent: {result}')

In [ ]:
print('=' * 60)
print('TEST 3 - Question about the video in Dutch')
print('=' * 60)
result = ask(
    'Welke fouten werden gemaakt tijdens het incident?',
    thread_id='test3',
    verbose=True
)
print(f'Agent: {result}')

In [ ]:
print('=' * 60)
print('TEST 4 - Ask for a summary in Dutch')
print('=' * 60)
result = ask(
    'Geef me een volledige samenvatting van de video in het Nederlands',
    thread_id='test4',
    verbose=True
)
print(f'Agent: {result}')

In [ ]:
print('=' * 60)
print('TEST 5 - Generate PDF cheatsheet')
print('=' * 60)
result = ask(
    'Maak een cheatsheet van de video in het Nederlands',
    thread_id='test5',
    verbose=True
)
print(f'Agent: {result}')

## 11. Test the agent - multi-tool chain

This is the killer demo test.
The user gives one natural language request that requires multiple tools in sequence.
The agent plans and executes all steps autonomously without any hardcoded routing.

In [ ]:
print('=' * 60)
print('TEST 6 - Multi-tool chain')
print('The agent chains multiple tools from a single request')
print('=' * 60)

result = ask(
    'Maak een samenvatting en een cheatsheet van de video '
    'en stuur die naar jan@example.be en piet@example.be',
    thread_id='test6',
    verbose=True
)
print(f'\nAgent final response:\n{result}')

## 12. Test conversation memory

The agent remembers the full conversation history within a thread.
This allows natural follow-up questions without repeating context.

In [ ]:
print('=' * 60)
print('TEST 7 - Conversation memory')
print('=' * 60)

# Turn 1 - load video
print('Turn 1:')
r1 = ask('https://www.youtube.com/watch?v=7OH5FEWWM_k', thread_id='memory_test', verbose=True)
print(f'Agent: {r1}\n')

# Turn 2 - question (agent remembers the video was loaded)
print('Turn 2:')
r2 = ask('Wat waren de drie grootste fouten?', thread_id='memory_test', verbose=True)
print(f'Agent: {r2}\n')

# Turn 3 - follow-up (agent remembers the previous answer)
print('Turn 3:')
r3 = ask('Geef meer detail over de eerste fout die je noemde', thread_id='memory_test', verbose=True)
print(f'Agent: {r3}')

## 13. Agent quality tests

Automated tests to verify the full agent works correctly before deployment.
These tests cover routing, memory, multilingual support and tool chaining.

In [ ]:
print('=' * 60)
print('AGENT QUALITY TESTS')
print('=' * 60)

tests_passed = 0
tests_failed = 0

def check(name, condition, detail=''):
    global tests_passed, tests_failed
    if condition:
        tests_passed += 1
        print(f'  PASS - {name}')
    else:
        tests_failed += 1
        print(f'  FAIL - {name}')
    if detail:
        print(f'         {detail}')

# Test 1 - Agent loads YouTube URL correctly
r = ask('https://www.youtube.com/watch?v=7OH5FEWWM_k', thread_id='qt1')
check('Agent detects YouTube URL and loads transcript',
    any(w in r.lower() for w in ['loaded', 'transcript', 'ready', 'video', 'geladen', 'klaar']),
    f'Response: {r[:80]}...')

# Test 2 - Agent answers English question
r = ask('What is this video about?', thread_id='qt2')
check('Agent answers English question in English',
    len(r) > 50 and any(w in r.lower() for w in ['fire', 'incident', 'brussels', 'building', 'training']),
    f'Response length: {len(r)} chars')

# Test 3 - Agent answers Dutch question in Dutch
r = ask('Waarover gaat deze video?', thread_id='qt3')
check('Agent answers Dutch question in Dutch',
    len(r) > 50 and any(w in r.lower() for w in ['brand', 'incident', 'brussel', 'gebouw', 'de', 'het', 'een']),
    f'Response length: {len(r)} chars')

# Test 4 - Agent generates summary when asked
r = ask('Geef me een samenvatting', thread_id='qt4')
check('Agent generates structured summary on request',
    len(r) > 200 and ('**' in r or '-' in r),
    f'Summary length: {len(r)} chars')

# Test 5 - Agent generates PDF when asked
r = ask('Maak een cheatsheet van de video', thread_id='qt5')
check('Agent generates PDF cheatsheet on request',
    any(w in r.lower() for w in ['pdf', 'cheatsheet', 'aangemaakt', 'generated', 'created']),
    f'Response: {r[:80]}...')

# Test 6 - Memory works across turns
ask('https://www.youtube.com/watch?v=7OH5FEWWM_k', thread_id='qt6')
r = ask('What did we just load?', thread_id='qt6')
check('Agent remembers previous messages in same thread',
    any(w in r.lower() for w in ['video', 'transcript', 'youtube', 'loaded', 'brussels']),
    f'Response: {r[:80]}...')

# Test 7 - Different thread has no memory of other thread
r = ask('What video did we load earlier?', thread_id='qt7_fresh')
check('Fresh thread has no memory of other conversations',
    any(w in r.lower() for w in ['no video', 'load', 'provide', 'geen video', 'url', 'first']),
    f'Response: {r[:80]}...')

# Test 8 - Agent handles out of scope question
r = ask('What is the capital of France?', thread_id='qt8')
check('Agent handles out-of-scope question gracefully',
    len(r) > 10,
    f'Response: {r[:80]}...')

print('\n' + '=' * 60)
print(f'RESULTS: {tests_passed} passed | {tests_failed} failed')
if tests_failed == 0:
    print('All tests passed - agent is ready for deployment!')
else:
    print('Fix the failing tests before deployment.')
print('=' * 60)

---

## What we built in this notebook

| Component | What | Why |
|-----------|------|-----|
| LangGraph graph | Two nodes connected by conditional edges | Enables autonomous tool selection |
| agent_node | LLM with all tools bound | Reads context and decides next action |
| tools_node | Executes the selected tool | Runs fetch, search, summarize, PDF or Gmail |
| tools_condition | Router between agent and tools | Loops until agent is satisfied |
| MemorySaver | Stores conversation per thread_id | Agent remembers context across turns |
| LangSmith | Traces all steps automatically | Live visualization during demo |

## What the agent can do now

All of these work from a single natural language message:
- Drop a YouTube URL -> video is loaded automatically
- Ask a question -> agent searches and answers
- Ask for a summary -> agent generates and returns it
- Ask for a cheatsheet -> agent creates the PDF
- Ask to send to emails -> agent sends PDF to those addresses
- Combine multiple requests -> agent chains tools in correct order
- Ask follow-up questions -> agent remembers the conversation

## Next: Notebook 05 - Evaluation
Evaluate the full agent using RAGAs, ROUGE and BLEU metrics.
Generate an evaluation report for the jury presentation.